In [ ]:
from modelscope import MsDataset
ds_dict = MsDataset.load('tzw20031220/CS-DeepSearchQA')
split_dict = MsDataset.load('tzw20031220/CS-DeepSearchQA', split='train')
print(ds_dict['train'][0])

In [10]:
import os
import re

def extract_log_section(content, filename):
    """提取日志部分内容"""
    lines = content.split('\n')
    in_section = False
    result = []
    
    for line in lines:
        # 检测日志标题
        if line == '## 🧪 实验/工作记录 (Log)':
            in_section = True
            result.append(f"## {filename}")
            continue
        
        # 如果遇到下一个标题，结束
        if in_section and line == '## 📊 结果与分析 (Analysis)':
            break
        
        if in_section:
            result.append(line)
    
    return '\n'.join(result) if result else None

def main():
    dir = "D:\\SciResearchLog\\10_Daily_Log"  # 修改为你的仓库路径
    output_file = "合并的日志.md"
    
    all_logs = []
    
    # 遍历所有 .md 文件
    for file in os.listdir(dir):
        if file.endswith('.md'):
            with open(os.path.join(dir, file), 'r', encoding='utf-8') as f:
                content = f.read()
            
            log_content = extract_log_section(content, file)
            if log_content:
                all_logs.append(log_content)
    
    # 写入合并文件
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("# 合并的实验/工作记录\n\n")
        f.write('\n---\n\n'.join(all_logs))
    
    print(f"已合并 {len(all_logs)} 个日志到 {output_file}")

if __name__ == "__main__":
    main()

已合并 14 个日志到 合并的日志.md


In [8]:
import json

input_file = r'D:\DissertationProject\DissertationProject\eval\eval_traj\BrowseComp-ZH_traj\SFT_IterScholar\20260406_220115_BrowseComp-ZH_traj_task42.jsonl'
output_file = r'D:\DissertationProject\DissertationProject\eval\eval_traj\BrowseComp-ZH_traj\SFT_IterScholar\20260406_220115_BrowseComp-ZH_traj_task42_decoded.jsonl'

def decode_text(text):
    """尝试将错误编码的文本解码为正确的 UTF-8"""
    try:
        # 先编码回字节（使用 Latin-1），再用 UTF-8 解码
        return text.encode('utf-8').decode('utf-8')
    except Exception as e:
        # 如果失败，返回原文本
        return text

decoded_lines = []

with open(input_file, 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        
        try:
            data = json.loads(line)
            
            # 递归解码所有字符串字段
            def decode_dict(obj):
                if isinstance(obj, dict):
                    return {k: decode_dict(v) for k, v in obj.items()}
                elif isinstance(obj, list):
                    return [decode_dict(item) for item in obj]
                elif isinstance(obj, str):
                    return decode_text(obj)
                else:
                    return obj
            
            decoded_data = decode_dict(data)
            decoded_lines.append(json.dumps(decoded_data, ensure_ascii=False))
            
            print(f"第 {line_num} 行已解码")
            
        except Exception as e:
            print(f"第 {line_num} 行处理失败: {e}")
            decoded_lines.append(line)  # 保留原始行

# 保存解码后的文件
with open(output_file, 'w', encoding='utf-8') as f:
    for line in decoded_lines:
        f.write(line + '\n')

print(f"\n解码完成！共处理 {len(decoded_lines)} 行")
print(f"输出文件: {output_file}")


第 1 行已解码
第 2 行已解码
第 3 行已解码

解码完成！共处理 3 行
输出文件: D:\DissertationProject\DissertationProject\eval\eval_traj\BrowseComp-ZH_traj\SFT_IterScholar\20260406_220115_BrowseComp-ZH_traj_task42_decoded.jsonl
